In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
folder = r'C:\Users\polti\OneDrive\Documents\FME\Trading\data'

df = pd.read_parquet(f'{folder}\\JPM.parquet')
print(df.columns)

df.rename(columns={'volumen': 'Volume'}, inplace=True)

# Agregar columna 'Adj Close' si no existe
if 'Adj Close' not in df.columns:
    df['Adj Close'] = pd.NA

# Lista de columnas en el orden deseado
ordered_cols = [
    'Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'RVT',
    'positivePartscr', 'negativePartscr', 'fearPartscr', 'findownPartscr',
    'finupPartscr', 'finhypePartscr', 'certaintyPartscr', 'uncertaintyPartscr'
]

# Agregar columnas extra (las que no están en ordered_cols) al final
extra_cols = [col for col in df.columns if col not in ordered_cols]
final_cols = ordered_cols + extra_cols

# Reordenar el DataFrame
df = df[final_cols]
# Crear columna de índice numérico
df.insert(0, 'Index', range(len(df)))

# Guardar como nuevo parquet (opcional)
df.to_parquet(r'C:\Users\polti\OneDrive\Documents\FME\Trading\data\JPM_reord.parquet', index=False)

In [ ]:
import yfinance as yf
import pandas as pd
import os
# Descargar datos de AAPL entre marzo 2012 y marzo 2014
aapl = yf.Ticker("HSBC")
hist = aapl.history(start="2012-01-03", end="2025-03-22").reset_index()

# Dejar solo las columnas deseadas
columns_of_interest = ['Date', 'Open', 'High', 'Low', 'Close']
hist_filtered = hist[columns_of_interest].copy()

# Convertir la columna Date al formato YYYY-MM-DD
hist_filtered['Date'] = pd.to_datetime(hist_filtered['Date']).dt.date

folder = r'C:\Users\polti\OneDrive\Documents\FME\Trading\data'
file_path = os.path.join(folder, 'HSBC_price.parquet')

# Guardar como archivo parquet
hist_filtered.to_parquet(file_path, index=False)



In [ ]:
import pandas as pd
df1 = pd.read_parquet(r'C:\Users\polti\OneDrive\Documents\FME\Trading\data\HSBC_sent.parquet')
df2 = pd.read_parquet(r'C:\Users\polti\OneDrive\Documents\FME\Trading\data\HSBC_price.parquet')

# Asegurarse de que la columna 'Date' esté en formato datetime
df1['Date'] = pd.to_datetime(df1['Date'])
df2['Date'] = pd.to_datetime(df2['Date'])

# Usar 'Date' como índice
df1.set_index('Date', inplace=True)
df2.set_index('Date', inplace=True)

# Hacer el inner join (solo fechas que aparecen en ambos)
merged = df2.join(df1, how='inner', lsuffix='_df1', rsuffix='_df2')
merged['Date'] = pd.to_datetime(merged.index).date
# Guardar el resultado
merged.to_parquet(r'C:\Users\polti\OneDrive\Documents\FME\Trading\data\HSBC.parquet', index=False)
print(merged.columns)

In [ ]:
print(merged.columns)

In [ ]:
import pandas as pd
#abrimos el parquet de DB_reord
df = pd.read_parquet(r'C:\Users\polti\OneDrive\Documents\FME\Trading\data\HSBC_reord.parquet')

df.drop(df.columns[0], axis=1, inplace=True)

# Establir la columna "Date" (o la corresponent) com a índex i convertir a datetime
df.set_index(df.columns[0], inplace=True)
df.index = pd.to_datetime(df.index)

# Crear les columnes de BULL, BEAR i finalment la BBr
df["BULL"] = 0.33 * (df["positivePartscr"] + df["certaintyPartscr"] + df["finhypePartscr"])
df["BEAR"] = 0.33 * (df["negativePartscr"] + df["uncertaintyPartscr"] + df["findownPartscr"])
df["BBr"] = 100 * df["BULL"] / (df["BULL"] + df["BEAR"])


#ver los valores de BBr de HSBC desde el 2023 al 2025
#ver tambien cuantos valores de BBr son superiores a 70
print(df.loc['2023-01-01':'2025-03-22', ['BBr']].head(10))
print("Valores de BBr superiores a 70:", (df['BBr'] > 74).sum())


: 

In [ ]:
df_70 = df.iloc[:int(len(df) * 0.7)]
print(df_70["BBr"].mean())
print(df_70["BBr"].min())
print(df_70["BBr"].max())
print(df_70["BBr"].std())
print(df_70["BBr"].median())
print(df_70["BBr"].quantile(0.25))
print(df_70["BBr"].quantile(0.75))

In [ ]:
import pandas as pd
df = pd.read_parquet(r'C:\Users\polti\OneDrive\Documents\FME\Trading\data\HSBC_reord.parquet')

#transformamos df a csv y guardamos en la misma carpeta que el parquet
df.to_csv(r'C:\Users\polti\OneDrive\Documents\FME\Trading\data\HSBC_reord.csv', index=False)

In [11]:
import pandas as pd
df = pd.read_csv(r'C:\Users\polti\Downloads\lexicons\lexicons.csv', encoding='utf-8')
print(df.columns)
#print(df.head())
print(df['Fear'].describe())


Index(['Id', 'AggDate', 'RCV', 'RVT', 'Positive', 'Negative', 'Fear',
       'FinDown', 'FinUp', 'FinHype', 'FinVolatile', 'Certainty',
       'Uncertainty', 'FinancialPolarity', 'PosNegRatio', 'PosNegLogRatio',
       'UpDownRatio', 'UpDownLogRatio', 'HypeVolRatio', 'HypeVolLogRatio',
       'CertUncertRatio', 'CertUncertLogRatio', 'AggregationType',
       'AggregationSubtype', 'DocAgg'],
      dtype='object')
count    791679.000000
mean         -0.098662
std           1.162099
min        -271.600000
25%          -0.000500
50%           0.000000
75%           0.000000
max           6.000000
Name: Fear, dtype: float64
